In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

root = "/content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake/xView"
print("Contenu de la racine MyDrive :")
for item in sorted(os.listdir(root)):
    print(" -", item)

Contenu de la racine MyDrive :
 - .ipynb_checkpoints
 - inference_demo
 - runs
 - tiles
 - train_images
 - train_images.tgz
 - train_labels
 - train_labels.tgz
 - training_data
 - val_images
 - val_images.tgz
 - yolo_labels


In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake/xView"

In [ ]:
import os

for sub in ["training_data", "yolo_labels", "runs", "tiles"]:
    path = os.path.join(DRIVE_ROOT, sub)
    print(f"\n--- {sub} ---")
    for item in sorted(os.listdir(path)):
        print(" -", item)


--- training_data ---
 - images
 - labels

--- yolo_labels ---
 - 10.txt
 - 100.txt
 - 102.txt
 - 1036.txt
 - 1037.txt
 - 104.txt
 - 1042.txt
 - 1044.txt
 - 1046.txt
 - 1048.txt
 - 1049.txt
 - 105.txt
 - 1050.txt
 - 1051.txt
 - 1052.txt
 - 1053.txt
 - 1056.txt
 - 1058.txt
 - 106.txt
 - 1061.txt
 - 1063.txt
 - 1065.txt
 - 1067.txt
 - 1068.txt
 - 107.txt
 - 1070.txt
 - 1072.txt
 - 1076.txt
 - 1077.txt
 - 1079.txt
 - 1080.txt
 - 1081.txt
 - 1084.txt
 - 1085.txt
 - 1086.txt
 - 1087.txt
 - 1088.txt
 - 1089.txt
 - 109.txt
 - 1090.txt
 - 1091.txt
 - 1092.txt
 - 1094.txt
 - 1095.txt
 - 110.txt
 - 1104.txt
 - 1106.txt
 - 1109.txt
 - 111.txt
 - 1114.txt
 - 1118.txt
 - 112.txt
 - 1120.txt
 - 1121.txt
 - 1124.txt
 - 1125.txt
 - 1126.txt
 - 1127.txt
 - 1128.txt
 - 1129.txt
 - 1132.txt
 - 1133.txt
 - 1135.txt
 - 1136.txt
 - 1139.txt
 - 1140.txt
 - 1141.txt
 - 1142.txt
 - 1149.txt
 - 1150.txt
 - 1154.txt
 - 1155.txt
 - 1158.txt
 - 1164.txt
 - 1165.txt
 - 1169.txt
 - 1175.txt
 - 1178.txt
 - 118.txt
 

In [ ]:
import os, shutil

drive_training_data = "/content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake/xView/training_data"
local_training_data = "/content/xView_local/training_data"

if not os.path.exists(local_training_data):
    print("Copie du dataset vers le disque local...")
    shutil.copytree(drive_training_data, local_training_data)
    print("Terminé.")
else:
    print("Déjà présent en local.")

for split in ["train", "val"]:
    n_img = len(os.listdir(os.path.join(local_training_data, "images", split)))
    print(f"{split} : {n_img} images")

Copie du dataset vers le disque local...
Terminé.
train : 10762 images
val : 1878 images


In [ ]:
import requests

url = "https://raw.githubusercontent.com/DIUx-xView/data_utilities/master/xview_class_labels.txt"
response = requests.get(url)

class_names = {}
for line in response.text.strip().split('\n'):
    idx, name = line.split(':')
    class_names[int(idx)] = name.strip()

official_ids = sorted(class_names.keys())
print(f"{len(official_ids)} classes chargées")
print("IDs officiels min/max :", min(official_ids), max(official_ids))

60 classes chargées
IDs officiels min/max : 11 94


In [ ]:
import glob

label_files = glob.glob(f"{local_training_data}/labels/train/*.txt") + glob.glob(f"{local_training_data}/labels/val/*.txt")
used_indices = set()
for lf in label_files:
    with open(lf) as f:
        for line in f:
            used_indices.add(int(line.split()[0]))

print("Indices min/max trouvés dans les labels :", min(used_indices), max(used_indices))
print("Nombre d'indices distincts :", len(used_indices))
print("Ces indices correspondent-ils aux IDs officiels ?", used_indices.issubset(set(official_ids)))

Indices min/max trouvés dans les labels : 0 59
Nombre d'indices distincts : 60
Ces indices correspondent-ils aux IDs officiels ? False


In [ ]:
sorted_official_ids = sorted(class_names.keys())  # 11 -> 94 avec des trous
id_to_contiguous = {orig_id: i for i, orig_id in enumerate(sorted_official_ids)}

class_names_ordered = [class_names[orig_id] for orig_id in sorted_official_ids]

print("Extrait début :", class_names_ordered[:5])
print("Extrait fin   :", class_names_ordered[-5:])
print("Total :", len(class_names_ordered))

Extrait début : ['Fixed-wing Aircraft', 'Small Aircraft', 'Cargo Plane', 'Helicopter', 'Passenger Vehicle']
Extrait fin   : ['Storage Tank', 'Shipping container lot', 'Shipping Container', 'Pylon', 'Tower']
Total : 60


In [ ]:
attendu_debut = ["Fixed-wing Aircraft", "Small Aircraft", "Cargo Plane", "Helicopter", "Passenger Vehicle"]
attendu_fin = ["Storage Tank", "Shipping container lot", "Shipping Container", "Pylon", "Tower"]

print("Début correspond :", class_names_ordered[:5] == attendu_debut)
print("Fin correspond    :", class_names_ordered[-5:] == attendu_fin)

Début correspond : True
Fin correspond    : True


In [ ]:
import yaml

data_yaml = {
    "path": local_training_data,
    "train": "images/train",
    "val": "images/val",
    "nc": len(class_names_ordered),
    "names": class_names_ordered
}

local_yaml_path = f"{local_training_data}/data.yaml"
with open(local_yaml_path, "w") as f:
    yaml.dump(data_yaml, f, sort_keys=False)

drive_yaml_path = f"{drive_training_data}/data.yaml"
shutil.copy(local_yaml_path, drive_yaml_path)

print("Sauvegardé en local et sur Drive :", drive_yaml_path)

Sauvegardé en local et sur Drive : /content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake/xView/training_data/data.yaml


In [ ]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 650.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 6.4 MB/s eta 0:00:00


In [ ]:
with open(local_yaml_path) as f:
    check = yaml.safe_load(f)
print("nc :", check["nc"])
print("5 premières classes :", check["names"][:5])

nc : 60
5 premières classes : ['Fixed-wing Aircraft', 'Small Aircraft', 'Cargo Plane', 'Helicopter', 'Passenger Vehicle']


In [ ]:
merge_map = {
    "Fixed-wing Aircraft": "Aircraft", "Small Aircraft": "Aircraft", "Cargo Plane": "Aircraft",
    "Pickup Truck": "Truck", "Utility Truck": "Truck", "Cargo Truck": "Truck",
    "Truck w/Box": "Truck", "Truck Tractor": "Truck", "Truck w/Flatbed": "Truck",
    "Truck w/Liquid": "Truck", "Crane Truck": "Truck", "Dump Truck": "Truck", "Haul Truck": "Truck",
    "Railway Vehicle": "Rail Vehicle", "Passenger Car": "Rail Vehicle", "Cargo Car": "Rail Vehicle",
    "Flat Car": "Rail Vehicle", "Tank car": "Rail Vehicle", "Locomotive": "Rail Vehicle",
    "Maritime Vessel": "Boat", "Motorboat": "Boat", "Sailboat": "Boat", "Tugboat": "Boat",
    "Barge": "Boat", "Fishing Vessel": "Boat", "Ferry": "Boat", "Yacht": "Boat",
    "Tower crane": "Construction Equipment", "Container Crane": "Construction Equipment",
    "Reach Stacker": "Construction Equipment", "Straddle Carrier": "Construction Equipment",
    "Mobile Crane": "Construction Equipment", "Scraper/Tractor": "Construction Equipment",
    "Cement Mixer": "Construction Equipment", "Ground Grader": "Construction Equipment",
}

new_names_full = [merge_map.get(n, n) for n in class_names_ordered]
new_unique_names = sorted(set(new_names_full))
name_to_new_idx = {n: i for i, n in enumerate(new_unique_names)}

old_idx_to_new_idx = {old_idx: name_to_new_idx[new_names_full[old_idx]] for old_idx in range(60)}

print(f"{len(new_unique_names)} classes après fusion (contre 60 avant)")
print(new_unique_names)

29 classes après fusion (contre 60 avant)
['Aircraft', 'Aircraft Hangar', 'Boat', 'Building', 'Bus', 'Construction Equipment', 'Construction Site', 'Container Ship', 'Damaged Building', 'Engineering Vehicle', 'Excavator', 'Facility', 'Front loader/Bulldozer', 'Helicopter', 'Helipad', 'Hut/Tent', 'Oil Tanker', 'Passenger Vehicle', 'Pylon', 'Rail Vehicle', 'Shed', 'Shipping Container', 'Shipping container lot', 'Small Car', 'Storage Tank', 'Tower', 'Trailer', 'Truck', 'Vehicle Lot']


In [ ]:
label_files = glob.glob(f"{local_training_data}/labels/train/*.txt") + glob.glob(f"{local_training_data}/labels/val/*.txt")

for lf in label_files:
    lines = open(lf).read().strip().splitlines()
    if not lines:
        continue
    new_lines = []
    for line in lines:
        parts = line.split()
        old_cls = int(parts[0])
        new_cls = old_idx_to_new_idx[old_cls]
        new_lines.append(" ".join([str(new_cls)] + parts[1:]))
    open(lf, "w").write("\n".join(new_lines) + "\n")

print(f"{len(label_files)} fichiers de labels remappés.")

12640 fichiers de labels remappés.


In [ ]:
data_yaml_merged = {
    "path": local_training_data,
    "train": "images/train",
    "val": "images/val",
    "nc": len(new_unique_names),
    "names": new_unique_names
}

local_yaml_merged_path = f"{local_training_data}/data_merged.yaml"
with open(local_yaml_merged_path, "w") as f:
    yaml.dump(data_yaml_merged, f, sort_keys=False)

drive_yaml_merged_path = f"{drive_training_data}/data_merged.yaml"
shutil.copy(local_yaml_merged_path, drive_yaml_merged_path)

print("data_merged.yaml sauvegardé :", drive_yaml_merged_path)

data_merged.yaml sauvegardé : /content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake/xView/training_data/data_merged.yaml


In [ ]:
from collections import Counter

counts = Counter()
for lf in label_files:
    with open(lf) as f:
        for line in f:
            counts[int(line.split()[0])] += 1

for idx, name in enumerate(new_unique_names):
    print(f"{name:25s} : {counts.get(idx, 0)}")

Aircraft                  : 1212
Aircraft Hangar           : 226
Boat                      : 4855
Building                  : 330794
Bus                       : 7134
Construction Equipment    : 1337
Construction Site         : 1330
Container Ship            : 352
Damaged Building          : 1149
Engineering Vehicle       : 212
Excavator                 : 870
Facility                  : 1038
Front loader/Bulldozer    : 645
Helicopter                : 78
Helipad                   : 127
Hut/Tent                  : 728
Oil Tanker                : 92
Passenger Vehicle         : 2999
Pylon                     : 396
Rail Vehicle              : 4006
Shed                      : 1227
Shipping Container        : 1610
Shipping container lot    : 2540
Small Car                 : 214591
Storage Tank              : 1789
Tower                     : 90
Trailer                   : 4188
Truck                     : 31003
Vehicle Lot               : 4599


In [ ]:
drive_labels_merged = f"{drive_training_data}/labels_merged"

for split in ["train", "val"]:
    src = f"{local_training_data}/labels/{split}"
    dst = f"{drive_labels_merged}/{split}"
    shutil.copytree(src, dst, dirs_exist_ok=True)

print("Labels fusionnés sauvegardés sur Drive :", drive_labels_merged)

Labels fusionnés sauvegardés sur Drive : /content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake/xView/training_data/labels_merged


In [23]:
import os

for split in ["train", "val"]:
    n_img_drive = len(os.listdir(f"{drive_training_data}/images/{split}"))
    n_lbl_merged_drive = len(os.listdir(f"{drive_training_data}/labels_merged/{split}"))
    print(f"[Drive] {split} : {n_img_drive} images, {n_lbl_merged_drive} labels_merged")

[Drive] train : 10762 images, 10762 labels_merged
[Drive] val : 1878 images, 1878 labels_merged


In [24]:
import glob

sample_file = glob.glob(f"{local_training_data}/labels/train/*.txt")[0]
max_idx = 0
for lf in glob.glob(f"{local_training_data}/labels/train/*.txt"):
    with open(lf) as f:
        for line in f:
            max_idx = max(max_idx, int(line.split()[0]))
print("Indice de classe maximum trouvé en local :", max_idx)

Indice de classe maximum trouvé en local : 28
